In [ ]:
import h5py
import plotly.express as px
import numpy as np
import scipy
import plotly.graph_objects as go
import pandas as pd
from calibrations import itof_calibration

file=r"J:\ctgroup\Edward\DATA\VMI\20250408\po_500mW_ell.cv4"
data={}
with h5py.File(file,'r') as f:
    for key in f.keys():
        data[key]=f[key][()]


cluster_data=pd.DataFrame({
    "pulse": data["cluster_corr"],
    "x": data["x"],
    "y": data["y"],
    "t": data["t"],
})

etof_data=pd.DataFrame({
    "pulse": data["etof_corr"],
    "etof": data["t_etof"],
})

itof_data=pd.DataFrame({
    "pulse": data["tof_corr"],
    "itof": data["t_tof"],
    "m/q": itof_calibration(data["t_tof"]),
})

In [ ]:
itof_data=itof_data[itof_data.itof.between(0,20000)]
zero_energy_pulses=cluster_data['pulse'][((cluster_data['x']-128)**2+(cluster_data['y']-128)**2)<10**2]

itof_zero_energy=itof_data[itof_data.pulse.isin(zero_energy_pulses)]

In [ ]:
px.histogram(itof_zero_energy, x="m/q", nbins=1000, histnorm="probability density", title="Zero Energy Gated", range_x=(0,60)).show()
px.histogram(itof_data, x="m/q", nbins=1000, histnorm="probability density", title="All Ions", range_x=(0,60)).show()

In [ ]:
multi_ion_hits= itof_data.groupby("pulse").filter(lambda x: len(x)>1).pulse.unique()

multi_ion_itof=itof_data[itof_data.pulse.isin(multi_ion_hits)]

px.histogram(multi_ion_itof, x="itof", nbins=1000, histnorm="density").show()

first_hits= multi_ion_itof.groupby("pulse").first()["m/q"]
last_hits= multi_ion_itof.groupby("pulse").last()["m/q"]

px.density_heatmap(x=first_hits, y=last_hits, nbinsx=256, nbinsy=256, width=800, height=800, title="Ion-Ion Coincidence", range_x=(0,60), range_y=(0,60)).show()

In [ ]:
first_hits_zero_energy=itof_zero_energy[itof_zero_energy.pulse.isin(multi_ion_hits)].groupby("pulse").first()["m/q"]
last_hits_zero_energy=itof_zero_energy[itof_zero_energy.pulse.isin(multi_ion_hits)].groupby("pulse").last()["m/q"]

px.density_heatmap(x=first_hits_zero_energy, y=last_hits_zero_energy, nbinsx=256, nbinsy=256, width=800, height=800, title="Ion-Ion Coincidence (Zero Energy)", range_x=(0,60), range_y=(0,60)).show()

In [ ]:
single_ion_hits=itof_data.groupby("pulse").filter(lambda x: len(x)==1).pulse.unique()
itof_single_hits= itof_data[itof_data.pulse.isin(single_ion_hits)]

px.histogram(itof_single_hits, x="m/q", nbins=1000, title="All ions (single ion per shot)").show()
itof_zero_energy_single=itof_zero_energy[itof_zero_energy.pulse.isin(single_ion_hits)]
px.histogram(itof_zero_energy_single, x="m/q", nbins=1000, title="Zero Energy Electron Gated (single ion per shot)").show()